# Practical Solution: Creating a Knowledge Graph in OWLReady2

In [ ]:
from owlready2 import *
# owlready2.JAVA_EXE = "C:\\path\\to\\java.exe" #windows users


We begin by importing an ontology. If we are extending an existing ontology then we would import this here. However, here we are creating an ontology from scratch, so we import a blank ontology.

In [ ]:
onto = get_ontology("http://www.dummy.info/new.owl")

We will first go through some examples of how to do things manually. Then we will realise that this is very tedious indeed and we will instead read the data in from an external file and create everything dynamically.

We start by creating some *classes* (types) of entity and attach these to the ontology. One way of doing this is use `with` which "opens" the ontology object we have just created and creates our new entities within it. Each entity is created by creating a new class of type `onto.Thing`.

In [ ]:

with onto: # This automatically attached the entities to the ontology
    class Staff(Thing):
        pass

    class Student(Thing):
        pass

    class Program(Thing):
        pass

    class Module(Thing):
        pass

    class School(Thing):
        pass

Now we can attach some attributes (properties to each of the entities). For each attribute, we will need to specify which type of entity the property should be attached to, and what the type of the allowable values is:

In [ ]:
with onto:

    class school_name(DataProperty):
        domain = [School]
        range = [str]

    class staff_id(DataProperty):
        domain = [Staff]
        range = [int]

    # Leave this one out initially to demonstrate reparenting
    class staff_title(DataProperty):
        domain = [Staff]
        range = [str]

    class student_id(DataProperty):
        domain = [Student]
        range = [int]

    class person_name(DataProperty):
        domain = [Staff,Student]
        range = [str]

    # Alternative way
    class program_title(Program >> str):
        pass

    class program_id(DataProperty):
        domain = [Module]
        range = [int]

    # Leave this one out initially to demonstrate reparenting
    class program_length(DataProperty):
        domain = [Program]
        range = [str]

    class module_title(DataProperty):
        domain = [Module]
        range = [str]

    class module_id(DataProperty):
        domain = [Module]
        range = [str]

Now we specify some relations. These are specified as `ObjectProperty` and must tell us what type of `Thing` they can be between:

In [ ]:
with onto:
    class offers_program(ObjectProperty):
        domain = [School]
        range = [Program]

    class has_module(ObjectProperty):
        domain = [Program]
        range = [Student]

    class is_enrolled_on(ObjectProperty):
        domain = [Student]
        range = [Program]

    class is_taught_by(ObjectProperty):
        domain = [Staff]
        range = [Module]

    class is_directed_by(ObjectProperty):
        domain = [Program]
        range = [Staff]

Let's save the ontology here

In [ ]:
onto.save('teaching.rdf')

## Populating the Graph

Now we populate the graph. As we are just exploring, we will only do this sparsely for now to see how it is done. We will then read everything in from files. We'll include:

* One School
* One programme
* One modules
* Two members of academic staff
* One student

Create the entities and assign their attributes

In [ ]:
eeecs = School(name='eeecs', school_name=['EEECS'])
mscaift = Program(name='mscaift', program_title = ['MSc AI Full-time'], program_id = [12345], program_length = ['1 year'])
knowledgeengineering = Module(name='knowledgeengineering', module_title = ["Knowledge Engineering"], module_id = ['ECS8052'])
iainstyles = Staff(name='iainstyles', person_name = ["Iain Styles"], staff_id = [894567], staff_title = ['Professor'])
# a different way to do it
barrydevereux = Staff(name='barrydevereux')
barrydevereux.person_name = ["Barry Devereux"]
barrydevereux.staff_id = [678945]
barrydevereux.staff_title = ['Dr']
alanturing = Student(name='alanturing', person_name = ['Alan Turing'], student_id = [234567])

Now we add in the relations to show how it's done

In [ ]:
eeecs.offers_program = [mscaift]
mscaift.has_module = [knowledgeengineering]
alanturing.is_enrolled_on = [mscaift]
knowledgeengineering.is_taught_by = [iainstyles]
mscaift.is_directed_by = [barrydevereux]


In [ ]:
onto.save('teaching.rdf')

## Querying the graph

Now we can construct some simple queries on the graph. 

In [ ]:
print(f"{knowledgeengineering.ModuleTitle[0]} is taught by {knowledgeengineering.is_taught_by[0].person_name[0]}")

This rapidly becomes inflexible: we want to query classes of object, and this will become very cumbersome. Fortunately there is a mechanism for this. The language designed for this is called SPARQL which is very similar to SQL. Let us see how it works with a few simple examples.

Here is a very simple query that returns everything in the dataset

In [ ]:
list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?subject ?predicate ?object
    WHERE{
        ?subject ?predicate ?object
    }
    """))


We can refine this query by, for example, restricting the predicate and the object to get specific object for which the predicate with variable object is true.

For example, to get all members of staff, we want to get all objects of type `QUBStaff`:

In [ ]:
list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?x
    WHERE{
        ?x rdf:type RDF:Staff
    }
    """))

Notice that this returns the *object* that satisfies the query.
Now get all students, this time printing the names:

In [ ]:
x = list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?student
    WHERE{
        ?student rdf:type RDF:Student
    }
    """))
print(x[0][0].person_name[0])

Get all modules and the staff who teach them

In [ ]:
list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?module
    WHERE{
        ?staff rdf:type ONTO:Staff
        ?module rdf:type ONTO:Module
        ?module ONTO:is_taught_by ?staff

    }
    """))

Our earlier query: all modules taught by an individual

In [ ]:
list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?module
    WHERE{
        ?module ONTO:is_taught_by ?staff
        ?staff ONTO:person_name "Iain Styles"
    }
    """))

Get all students taught by each member of staff

In [ ]:
list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?student
    WHERE{
        ?staff rdf:type ONTO:Staff
        ?student rdf:type ONTO:Student
        ?program rdf:type ONTO:Program
        ?module rdf:type ONTO:Module
        ?module ONTO:is_taught_by ?staff
        ?program ONTO:has_module ?module
        ?student ONTO:is_enrolled_on ?program

    }
    """))

This is somewhat limiting and we need a bigger set of facts to work with. We'll load them in from external files. This is often the first - and sometime most complex part of the project because quite often the graph is not in RDF format and instead comes as a set of CSV/JSON files. In our case we have six files to work with:

* `Schools.csv` lists the entities of type `school` and their attributes.
* `Programs.csv`
* `Modules.csv`
* `Staff.csv`
* `Students.csv`
* `Graph.csv` lists the triples that define the graph

To avoid confusion we will clear all the variables so - **please restart the interpreter**.

We will also use pandas to load the data in as it allows to read in the column headers more easily.

In [1]:
from owlready2 import *
import pandas as pd
entity_list = ['School','Staff','Program','Module','Student']
onto = get_ontology("http://www.dummy.info/new.owl")

# Load the data in from the various files. We will first load all the data in before creating the objects



entities = dict()
for entity in entity_list:
        print(f'Loading data of type {entity}')
        with open(f'{entity}.csv') as csvfile:
            entities[entity] = pd.read_csv(csvfile)

Loading data of type School
Loading data of type Staff
Loading data of type Program
Loading data of type Module
Loading data of type Student


In [2]:
# Create the entities

with onto:
    EntityClasses = dict()
    for entity in entities.keys():
        EntityClasses[entity] = type(entity, (Thing,), dict())
        print(f'Created entity class {EntityClasses[entity]}')

Created entity class new.School
Created entity class new.Staff
Created entity class new.Program
Created entity class new.Module
Created entity class new.Student


In [3]:

# Now sort out the attributes. We have to extract, for each attribute, what its domain and range are
with onto:
    attributes = dict()
    for entity in entities.keys():
        attribute_names = list(entities[entity])
        for a in attribute_names[1:]:
                if a not in attributes.keys():
                    attributes[a] = {'domain': [EntityClasses[entity]], 'range': [str]}# will work for everything we have here, but will need to more sophisticated for more complex models
                else:
                    attributes[a]['domain'].append(EntityClasses[entity])
    print(attributes)

{'school_name': {'domain': [new.School], 'range': [<class 'str'>]}, 'person_name': {'domain': [new.Staff, new.Student], 'range': [<class 'str'>]}, 'person_id': {'domain': [new.Staff, new.Student], 'range': [<class 'str'>]}, 'program_title': {'domain': [new.Program], 'range': [<class 'str'>]}, 'program_id': {'domain': [new.Program], 'range': [<class 'str'>]}, 'program_length': {'domain': [new.Program], 'range': [<class 'str'>]}, 'module_title': {'domain': [new.Module], 'range': [<class 'str'>]}, 'module_id': {'domain': [new.Module], 'range': [<class 'str'>]}}


In [4]:

# Create the attribute classes
with onto:
    print(attributes)

    AttributeClasses = dict()
    for a in attributes.keys():
        AttributeClasses[a] = type(a, (DataProperty,), attributes[a])
        print(f'Created attribute {AttributeClasses[a]} with properties {attributes[a]}')

{'school_name': {'domain': [new.School], 'range': [<class 'str'>]}, 'person_name': {'domain': [new.Staff, new.Student], 'range': [<class 'str'>]}, 'person_id': {'domain': [new.Staff, new.Student], 'range': [<class 'str'>]}, 'program_title': {'domain': [new.Program], 'range': [<class 'str'>]}, 'program_id': {'domain': [new.Program], 'range': [<class 'str'>]}, 'program_length': {'domain': [new.Program], 'range': [<class 'str'>]}, 'module_title': {'domain': [new.Module], 'range': [<class 'str'>]}, 'module_id': {'domain': [new.Module], 'range': [<class 'str'>]}}
Created attribute new.school_name with properties {'_name': 'school_name', 'namespace': get_ontology("http://www.dummy.info/new.owl#"), 'storid': 308, 'is_a': [owl.DatatypeProperty], '_equivalent_to': None}
Created attribute new.person_name with properties {'_name': 'person_name', 'namespace': get_ontology("http://www.dummy.info/new.owl#"), 'storid': 309, 'is_a': [owl.DatatypeProperty], '_equivalent_to': None}
Created attribute new

In [5]:

# Now we populate the classes
with onto:
    # First the entities
    nodes = dict()
    for entity in entities.keys():
        for i in entities[entity].iterrows():
            x = i[1].to_dict()
            nodes[x['name']] = EntityClasses[entity](name=x['name'])
            for i in x.keys():
                if i == 'name':
                    continue
                else:
                    getattr(nodes[x['name']],i).append(x[i])
            print(f"Created node {x['name']} of type {entity} with properties {[getattr(nodes[x['name']],i) for i in x.keys()]}")



Created node sch01 of type School with properties ['sch01', ['EEECS']]
Created node sch02 of type School with properties ['sch02', ['Maths and Physics']]
Created node sta01 of type Staff with properties ['sta01', ['Dr Barry Devereux'], [12345]]
Created node sta02 of type Staff with properties ['sta02', ['Dr Ihsen Alouani'], [12346]]
Created node sta03 of type Staff with properties ['sta03', ['Dr Lu Bai'], [12347]]
Created node sta04 of type Staff with properties ['sta04', ['Prof Hui Wang'], [12348]]
Created node sta05 of type Staff with properties ['sta05', ['Prof Iain Styles'], [12349]]
Created node sta06 of type Staff with properties ['sta06', ['Dr Yang Hua'], [12350]]
Created node sta07 of type Staff with properties ['sta07', ['Dr Barry Devereux'], [12351]]
Created node sta08 of type Staff with properties ['sta08', ['Dr Reza Rafiee'], [12352]]
Created node sta09 of type Staff with properties ['sta09', ['Dr Niall McLaughlin'], [12353]]
Created node sta10 of type Staff with properties

In [6]:
# Finally the relations
with open('Relations.csv') as csvfile:
     relationstable = pd.read_csv(csvfile)
     print(relationstable)


      src          relation   dest
0   sch01  offers_programme  pro01
1   sch01  offers_programme  pro02
2   sch02  offers_programme  pro03
3   pro01        has_module  mod01
4   pro01        has_module  mod02
5   pro01        has_module  mod03
6   pro01        has_module  mod04
7   pro01        has_module  mod05
8   pro01        has_module  mod06
9   pro01        has_module  mod07
10  pro02        has_module  mod06
11  pro02        has_module  mod08
12  pro02        has_module  mod09
13  pro02        has_module  mod10
14  pro02        has_module  mod11
15  pro02        has_module  mod12
16  pro02        has_module  mod02
17  pro03        has_module  mod04
18  pro03        has_module  mod05
19  pro03        has_module  mod13
20  pro03        has_module  mod14
21  pro03        has_module  mod02
22  pro03        has_module  mod15
23  pro03        has_module  mod16
24  pro01    is_directed_by  sta01
25  pro02    is_directed_by  sta02
26  pro03    is_directed_by  sta03
27  mod01      is_ta

In [7]:
with onto:
    relations = dict()
    for row in relationstable.iterrows():
        rel = row[1]['relation']
        domain = type(nodes[row[1]['src']]).__name__
        range = type(nodes[row[1]['dest']]).__name__
        if rel not in relations.keys():
            relations[rel] = {'domain':[EntityClasses[domain]], 'range':[EntityClasses[range]]}
        else:
            relations[rel]['domain'].append(EntityClasses[domain])
            relations[rel]['range'].append(EntityClasses[range])
    print(relations)

    for k in relations.keys():
        relations[k]['domain'] = list(set(relations[k]['domain']))
        relations[k]['range'] = list(set(relations[k]['range']))


    print(relations)


{'offers_programme': {'domain': [new.School, new.School, new.School], 'range': [new.Program, new.Program, new.Program]}, 'has_module': {'domain': [new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program, new.Program], 'range': [new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module]}, 'is_directed_by': {'domain': [new.Program, new.Program, new.Program], 'range': [new.Staff, new.Staff, new.Staff]}, 'is_taught_by': {'domain': [new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.Module, new.

In [8]:
with onto:
    RelationClasses = dict()
    for r in relations.keys():
        RelationClasses[r] = type(r, (ObjectProperty,), relations[r])
        print(f"Created relation class {RelationClasses[r]} of type {r}")
        

Created relation class new.offers_programme of type offers_programme
Created relation class new.has_module of type has_module
Created relation class new.is_directed_by of type is_directed_by
Created relation class new.is_taught_by of type is_taught_by
Created relation class new.is_enrolled_on of type is_enrolled_on


In [9]:
# Now we populate
with onto:
    for row in relationstable.iterrows():
        x = row[1].to_dict()
        getattr(nodes[x['src']],x['relation']).append(nodes[x['dest']])

Let's now run some of the queries again

In [14]:
x = list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?x
    WHERE{
        ?x rdf:type RDF:Staff
    }
    """))

for i in x:
    print(i[0].person_name)

['Dr Barry Devereux']
['Dr Ihsen Alouani']
['Dr Lu Bai']
['Prof Hui Wang']
['Prof Iain Styles']
['Dr Yang Hua']
['Dr Barry Devereux']
['Dr Reza Rafiee']
['Dr Niall McLaughlin']
['Dr Ciara Rafferty']
['Dr Ayesha Khalid']
['Dr Amy Liu']
["Prof Maire O'Neill"]
['Dr Lisa McFetridge']
['Dr Hannah Mitchell']
['Dr Neil Anderson']
['Dr Chao Tian']


In [29]:
x = list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?student
    WHERE{
        ?student rdf:type RDF:Student
    }
    """))
for i in x:
    print(i[0].person_name)

['Alan Turing']
['John McCarthy']
['Raj Reddy']
['Geoffrey Hinton']
['Demis Hassabis']
['Annie Easley']
['Martin Hellman']
['Whitfield Diffie']
['Margaret Hamilton']
['Peter Shor']
['John von Neumann']
['Norbert Wiener']
['Sophie Wilson']
['Edgar Codd']
['Grace Hopper']


In [17]:
x = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?module
    WHERE{
        ?staff rdf:type ONTO:Staff
        ?module rdf:type ONTO:Module
        ?module ONTO:is_taught_by ?staff

    }
    """))

for i in x:
    print(f"{i[0].person_name[0]} teaches {i[1].module_title[0]}")

Dr Barry Devereux teaches Natural Language Processing
Dr Ihsen Alouani teaches Introduction to AI
Dr Ihsen Alouani teaches AI for Cyber Security
Prof Hui Wang teaches Machine Learning
Prof Iain Styles teaches Knowledge Engineering
Dr Yang Hua teaches Computer Vision
Dr Barry Devereux teaches AI for Health
Dr Reza Rafiee teaches Malware Detection
Dr Niall McLaughlin teaches Cryptography
Dr Ciara Rafferty teaches Penetration Testing
Dr Ayesha Khalid teaches Secure Information Systems
Dr Amy Liu teaches Hardware Security
Prof Maire O'Neill teaches Statistics
Dr Lisa McFetridge teaches Data Visualisation
Dr Hannah Mitchell teaches Databases
Dr Neil Anderson teaches Data Engineering


In [18]:
list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?student
    WHERE{
        ?staff rdf:type ONTO:Staff
        ?student rdf:type ONTO:Student
        ?program rdf:type ONTO:Program
        ?module rdf:type ONTO:Module
        ?module ONTO:is_taught_by ?staff
        ?program ONTO:has_module ?module
        ?student ONTO:is_enrolled_on ?program

    }
    """))

[[new.sta01, new.stu01],
 [new.sta01, new.stu02],
 [new.sta01, new.stu03],
 [new.sta01, new.stu04],
 [new.sta01, new.stu05],
 [new.sta01, new.stu11],
 [new.sta01, new.stu12],
 [new.sta01, new.stu13],
 [new.sta01, new.stu14],
 [new.sta01, new.stu15],
 [new.sta02, new.stu01],
 [new.sta02, new.stu01],
 [new.sta02, new.stu02],
 [new.sta02, new.stu02],
 [new.sta02, new.stu03],
 [new.sta02, new.stu03],
 [new.sta02, new.stu04],
 [new.sta02, new.stu04],
 [new.sta02, new.stu05],
 [new.sta02, new.stu05],
 [new.sta02, new.stu06],
 [new.sta02, new.stu07],
 [new.sta02, new.stu08],
 [new.sta02, new.stu09],
 [new.sta02, new.stu10],
 [new.sta04, new.stu01],
 [new.sta04, new.stu02],
 [new.sta04, new.stu03],
 [new.sta04, new.stu04],
 [new.sta04, new.stu05],
 [new.sta04, new.stu06],
 [new.sta04, new.stu07],
 [new.sta04, new.stu08],
 [new.sta04, new.stu09],
 [new.sta04, new.stu10],
 [new.sta04, new.stu11],
 [new.sta04, new.stu12],
 [new.sta04, new.stu13],
 [new.sta04, new.stu14],
 [new.sta04, new.stu15],


Finally, let's extract and visualise both the graph and the ontology.

In [ ]:
# Placeholder for code 